# 02 — Negative Binomial Production Function

Notebook ini membangun model fungsi produksi lulusan perguruan tinggi pada unit:

\[
\text{provinsi} \times \text{kabupaten/kota} \times
\text{jenis kelamin} \times \text{jenjang}.
\]

Outcome utama adalah `jumlah_lulusan`, sedangkan `jumlah_terdaftar` digunakan
sebagai **exposure/offset**:

\[
Y_i \sim \operatorname{NB2}(\mu_i,\alpha), \qquad
\log(\mu_i)=\log(E_i)+X_i\beta.
\]

Dengan demikian, koefisien menjelaskan perubahan **expected graduate output
per registered-student stock**, bukan probabilitas individual untuk lulus.

## Tujuan

1. Membuktikan bahwa Poisson tidak memadai akibat overdispersion.
2. Mengestimasi rangkaian model Negative Binomial secara bertingkat.
3. Melaporkan *incidence rate ratio* (IRR) dengan standard error yang
   dikelompokkan menurut provinsi.
4. Memeriksa multikolinearitas, kalibrasi, residual, dan pencilan.
5. Menjalankan analisis sensitivitas yang telah ditetapkan.
6. Menghasilkan tabel, model tersimpan, dan gambar siap artikel.

> **Batas inferensi:** data bersifat agregat, ekologis, dan cross-sectional.
> Hubungan yang ditemukan tidak membuktikan sebab-akibat dan tidak dapat
> dipindahkan langsung ke tingkat mahasiswa individual.


In [1]:
from __future__ import annotations

import json
import platform
import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from scipy import stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_poisson_deviance,
    mean_squared_error,
    mean_squared_log_error,
)
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

print("Python      :", sys.version.split()[0])
print("Pandas      :", pd.__version__)
print("NumPy       :", np.__version__)
print("SciPy       :", scipy.__version__)
print("Statsmodels :", statsmodels.__version__)
print("Platform    :", platform.platform())


/home/oai/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-15i7pc83 because there was an issue with the default path (/home/oai/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Python      : 3.13.5
Pandas      : 2.2.3
NumPy       : 2.3.5
SciPy       : 1.17.0
Statsmodels : 0.14.6
Platform    : Linux-6.12.13-x86_64-with-glibc2.41


In [2]:
# Resolusi lokasi proyek yang robust.
CURRENT_DIR = Path.cwd().resolve()
WORKBOOK_NAME = "Dataset_Gabungan_Kemdiktisaintek_2025.xlsx"


def candidate_project_roots(start: Path) -> list[Path]:
    candidates: list[Path] = []

    def add(path: Path) -> None:
        try:
            resolved = path.resolve()
        except OSError:
            return
        if resolved not in candidates:
            candidates.append(resolved)

    add(start)
    for parent in start.parents:
        add(parent)

    try:
        for child in start.iterdir():
            if child.is_dir():
                add(child)
    except (PermissionError, FileNotFoundError):
        pass

    common_names = [
        "pt_education_research_project",
        "pt_education_research_project_00_01",
        "pt_education_research_project_00_01_v2",
        "pt_education_research_project_00_01_02",
    ]
    for base in list(candidates):
        for name in common_names:
            add(base / name)

    return candidates


PROJECT_ROOT = None
COMBINED_FILE = None

for candidate in candidate_project_roots(CURRENT_DIR):
    workbook_candidate = candidate / "data" / "processed" / WORKBOOK_NAME
    if workbook_candidate.exists():
        PROJECT_ROOT = candidate
        COMBINED_FILE = workbook_candidate
        break

if COMBINED_FILE is None:
    for candidate in [CURRENT_DIR, CURRENT_DIR.parent]:
        workbook_candidate = candidate / WORKBOOK_NAME
        if workbook_candidate.exists():
            PROJECT_ROOT = candidate
            COMBINED_FILE = workbook_candidate
            break

if PROJECT_ROOT is None or COMBINED_FILE is None:
    searched = "\n".join(
        str(path / "data" / "processed" / WORKBOOK_NAME)
        for path in candidate_project_roots(CURRENT_DIR)[:15]
    )
    raise FileNotFoundError(
        "Dataset gabungan tidak ditemukan. Ekstrak paket proyek lengkap, "
        "kemudian jalankan Jupyter dari folder proyek. Lokasi yang diperiksa:\n"
        + searched
    )

OUTPUT_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUT_MODEL_DIR = PROJECT_ROOT / "outputs" / "models"

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

VALIDATED_GENDER_FILE = OUTPUT_TABLE_DIR / "00_validated_AD_Gender_Jenjang.csv"
READINESS_00_FILE = OUTPUT_TABLE_DIR / "00_readiness_checklist.csv"
READINESS_01_FILE = OUTPUT_TABLE_DIR / "01_readiness_checklist.csv"

print("CURRENT_DIR  :", CURRENT_DIR)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("COMBINED_FILE:", COMBINED_FILE)


CURRENT_DIR  : /mnt/data/work_project02/pt_education_research_project
PROJECT_ROOT : /mnt/data/work_project02/pt_education_research_project
COMBINED_FILE: /mnt/data/work_project02/pt_education_research_project/data/processed/Dataset_Gabungan_Kemdiktisaintek_2025.xlsx


In [3]:
# Fungsi ditanamkan langsung agar notebook mandiri.

def standardize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result.columns = (
        pd.Index(result.columns)
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    return result


def safe_divide(numerator, denominator):
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")
    return num.div(den.where(den.ne(0)))


def recompute_gender_indicators(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()

    count_columns = [
        "jumlah_lulusan", "jumlah_terdaftar", "jumlah_aktif_a",
        "jumlah_mbkm_m", "jumlah_pt", "jumlah_pt_akreditasi_tinggi",
        "dosen_total_jabatan", "dosen_profesor", "dosen_senior",
        "dosen_total_sertifikasi", "dosen_tersertifikasi",
    ]
    for column in count_columns:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column], errors="coerce")

    result["jumlah_aktif_total"] = result[
        ["jumlah_aktif_a", "jumlah_mbkm_m"]
    ].sum(axis=1, min_count=1)

    result["graduate_output_intensity"] = safe_divide(
        result["jumlah_lulusan"], result["jumlah_terdaftar"]
    )
    result["lulusan_per_1000_terdaftar"] = (
        result["graduate_output_intensity"] * 1000
    )
    result["active_share"] = safe_divide(
        result["jumlah_aktif_total"], result["jumlah_terdaftar"]
    )
    result["mbkm_share"] = safe_divide(
        result["jumlah_mbkm_m"], result["jumlah_aktif_total"]
    )
    result["pt_high_share"] = safe_divide(
        result["jumlah_pt_akreditasi_tinggi"], result["jumlah_pt"]
    )
    result["prof_share"] = safe_divide(
        result["dosen_profesor"], result["dosen_total_jabatan"]
    )
    result["senior_share"] = safe_divide(
        result["dosen_senior"], result["dosen_total_jabatan"]
    )
    result["cert_share"] = safe_divide(
        result["dosen_tersertifikasi"], result["dosen_total_sertifikasi"]
    )
    result["log_pt"] = np.log1p(result["jumlah_pt"])
    return result


def zscore_with_parameters(frame: pd.DataFrame, columns: list[str]):
    result = frame.copy()
    records = []
    for column in columns:
        mean = float(result[column].mean())
        sd = float(result[column].std(ddof=0))
        if not np.isfinite(sd) or sd <= 0:
            raise ValueError(f"Standard deviation tidak valid: {column}={sd}")
        result[f"z_{column}"] = (result[column] - mean) / sd
        records.append({
            "variable": column,
            "mean": mean,
            "std_population": sd,
            "minimum": float(result[column].min()),
            "maximum": float(result[column].max()),
        })
    return result, pd.DataFrame(records)


def fit_nb2_clustered(formula: str, data: pd.DataFrame, cluster_column: str = "provinsi_pt", maxiter: int = 400):
    model = NegativeBinomial.from_formula(
        formula=formula,
        data=data,
        offset=np.log(data["jumlah_terdaftar"]),
        loglike_method="nb2",
    )
    return model.fit(
        disp=False,
        maxiter=maxiter,
        cov_type="cluster",
        cov_kwds={"groups": data[cluster_column], "use_correction": True},
    )


def fit_nb2_hc1(formula: str, data: pd.DataFrame, maxiter: int = 400):
    model = NegativeBinomial.from_formula(
        formula=formula,
        data=data,
        offset=np.log(data["jumlah_terdaftar"]),
        loglike_method="nb2",
    )
    return model.fit(disp=False, maxiter=maxiter, cov_type="HC1")


def model_comparison_row(name: str, result) -> dict:
    return {
        "model": name,
        "nobs": int(result.nobs),
        "df_model": float(result.df_model),
        "df_resid": float(result.df_resid),
        "log_likelihood": float(result.llf),
        "aic": float(result.aic),
        "bic": float(result.bic),
        "mcfadden_pseudo_r2": float(result.prsquared),
        "alpha": float(result.params.get("alpha", np.nan)),
        "converged": bool(result.mle_retvals.get("converged", False)),
        "covariance": str(result.cov_type),
    }


def likelihood_ratio_test(reduced_name: str, reduced_result, full_name: str, full_result) -> dict:
    lr_stat = 2 * (full_result.llf - reduced_result.llf)
    df_difference = int(round(full_result.df_model - reduced_result.df_model))
    p_value = stats.chi2.sf(lr_stat, df_difference) if df_difference > 0 else np.nan
    return {
        "reduced_model": reduced_name,
        "full_model": full_name,
        "lr_statistic": float(lr_stat),
        "df_difference": df_difference,
        "p_value": float(p_value),
        "significant_0_05": bool(p_value < 0.05) if np.isfinite(p_value) else False,
    }


def extract_irr_table(result, model_name: str) -> pd.DataFrame:
    conf = result.conf_int()
    table = pd.DataFrame({
        "model": model_name,
        "term": result.params.index,
        "coefficient": result.params.to_numpy(),
        "std_error": result.bse.to_numpy(),
        "z_value": result.tvalues.to_numpy(),
        "p_value": result.pvalues.to_numpy(),
        "ci_lower_coefficient": conf.iloc[:, 0].to_numpy(),
        "ci_upper_coefficient": conf.iloc[:, 1].to_numpy(),
    })
    table["irr"] = np.exp(table["coefficient"])
    table["irr_ci_lower"] = np.exp(table["ci_lower_coefficient"])
    table["irr_ci_upper"] = np.exp(table["ci_upper_coefficient"])
    table["significant_0_05"] = table["p_value"] < 0.05
    return table


def prediction_metrics(y_true, y_pred, alpha: float) -> dict:
    y = np.asarray(y_true, dtype=float)
    pred = np.clip(np.asarray(y_pred, dtype=float), 1e-12, None)
    nb_family = sm.families.NegativeBinomial(alpha=float(alpha))
    return {
        "mae": float(mean_absolute_error(y, pred)),
        "rmse": float(mean_squared_error(y, pred) ** 0.5),
        "rmsle": float(mean_squared_log_error(y, pred) ** 0.5),
        "mean_poisson_deviance": float(mean_poisson_deviance(y, pred)),
        "mean_nb_deviance": float(nb_family.deviance(y, pred) / len(y)),
        "observed_total": float(y.sum()),
        "predicted_total": float(pred.sum()),
        "predicted_to_observed_ratio": float(pred.sum() / y.sum()),
    }


def calibration_by_quantile(frame: pd.DataFrame, observed_column: str, predicted_column: str, q: int = 10):
    result = frame.copy()
    ranked = result[predicted_column].rank(method="first")
    result["prediction_decile"] = pd.qcut(ranked, q=q, labels=False, duplicates="drop") + 1
    calibration = result.groupby("prediction_decile", as_index=False).agg(
        n=(observed_column, "size"),
        observed_total=(observed_column, "sum"),
        predicted_total=(predicted_column, "sum"),
        observed_mean=(observed_column, "mean"),
        predicted_mean=(predicted_column, "mean"),
    )
    calibration["observed_to_predicted_ratio"] = safe_divide(
        calibration["observed_total"], calibration["predicted_total"]
    )
    return calibration


def save_figure(fig, filename: str) -> Path:
    path = OUTPUT_FIGURE_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return path


## 1. Memuat dan memvalidasi dataset

Notebook memprioritaskan CSV tervalidasi hasil Notebook 00. Jika file itu tidak
tersedia, workbook gabungan digunakan sebagai fallback.

Notebook juga memastikan bahwa Notebook 00 dan 01 telah berstatus `PASS`
apabila checklist-nya tersedia.


In [4]:
if VALIDATED_GENDER_FILE.exists():
    DATA_SOURCE = "CSV tervalidasi hasil Notebook 00"
    gender_df = pd.read_csv(VALIDATED_GENDER_FILE)
else:
    DATA_SOURCE = "Workbook gabungan — fallback"
    gender_df = pd.read_excel(
        COMBINED_FILE,
        sheet_name="AD_Gender_Jenjang",
        engine="openpyxl",
    )

gender_df = recompute_gender_indicators(standardize_columns(gender_df))

KEY_A = ["provinsi_pt", "kabupaten_pt", "jk", "nm_jenj_didik"]
REQUIRED_COLUMNS = {
    *KEY_A,
    "jumlah_lulusan", "jumlah_terdaftar", "jumlah_aktif_a", "jumlah_mbkm_m",
    "jumlah_pt", "jumlah_pt_akreditasi_tinggi", "dosen_total_jabatan",
    "dosen_profesor", "dosen_senior", "dosen_total_sertifikasi",
    "dosen_tersertifikasi", "flag_match_terdaftar", "flag_match_aktif",
    "flag_match_pt", "flag_match_dosen",
}
missing_columns = sorted(REQUIRED_COLUMNS - set(gender_df.columns))
assert not missing_columns, f"Kolom wajib tidak tersedia: {missing_columns}"
assert not gender_df.duplicated(KEY_A).any(), "Kunci Dataset A tidak unik."
assert gender_df["jumlah_lulusan"].ge(0).all(), "Outcome negatif ditemukan."

for readiness_path, notebook_name in [
    (READINESS_00_FILE, "Notebook 00"),
    (READINESS_01_FILE, "Notebook 01"),
]:
    if readiness_path.exists():
        readiness = pd.read_csv(readiness_path)
        assert readiness["status"].eq("PASS").all(), f"{notebook_name} belum seluruhnya PASS."

print("Data source:", DATA_SOURCE)
print("Rows       :", f"{len(gender_df):,}")
print("Columns    :", len(gender_df.columns))
print("PASS — validasi minimum terpenuhi.")


Data source: Workbook gabungan — fallback
Rows       : 2,624
Columns    : 40
PASS — validasi minimum terpenuhi.


## 2. Menetapkan sampel model dan transformasi prediktor

Semua model M0–M5 diestimasi pada **sampel complete-case yang sama** agar AIC,
BIC, dan likelihood-ratio test dapat dibandingkan secara sah.

Prediktor numerik diubah menjadi skor-z. IRR untuk prediktor numerik karena itu
diinterpretasikan sebagai perubahan expected output untuk kenaikan satu standar
deviasi prediktor, dengan variabel lain tetap.


In [5]:
for column in ["provinsi_pt", "kabupaten_pt", "jk", "nm_jenj_didik"]:
    gender_df[column] = (
        gender_df[column].astype("string").str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True).str.strip()
    )

NUMERIC_PREDICTORS_RAW = [
    "active_share", "mbkm_share", "log_pt", "pt_high_share",
    "prof_share", "cert_share", "senior_share",
]
MODEL_REQUIRED = [
    "jumlah_lulusan", "jumlah_terdaftar", "provinsi_pt", "kabupaten_pt",
    "jk", "nm_jenj_didik", *NUMERIC_PREDICTORS_RAW,
]

sample_flow = [{"stage": "Dataset A awal", "n_rows": len(gender_df), "excluded_from_previous": 0}]
model_df = gender_df.dropna(subset=MODEL_REQUIRED).copy()
sample_flow.append({
    "stage": "Complete-case seluruh prediktor utama dan sensitivitas",
    "n_rows": len(model_df),
    "excluded_from_previous": len(gender_df) - len(model_df),
})
before_positive = len(model_df)
model_df = model_df.loc[(model_df["jumlah_terdaftar"] > 0) & (model_df["jumlah_lulusan"] >= 0)].copy()
sample_flow.append({
    "stage": "Exposure positif dan outcome nonnegatif",
    "n_rows": len(model_df),
    "excluded_from_previous": before_positive - len(model_df),
})

MATCH_FLAGS = ["flag_match_terdaftar", "flag_match_aktif", "flag_match_pt", "flag_match_dosen"]
model_df["all_sources_match"] = model_df[MATCH_FLAGS].eq("MATCH").all(axis=1)
assert model_df["all_sources_match"].all(), "Sampel model masih mengandung baris dengan sumber tidak cocok."
assert model_df["jk"].isin(["L", "P"]).all()
assert model_df["provinsi_pt"].nunique() >= 30

model_df, scaling_parameters = zscore_with_parameters(model_df, NUMERIC_PREDICTORS_RAW)
sample_flow = pd.DataFrame(sample_flow)
sample_flow["retained_pct_of_initial"] = sample_flow["n_rows"] / len(gender_df) * 100

sample_flow.to_csv(OUTPUT_TABLE_DIR / "02_sample_flow.csv", index=False)
scaling_parameters.to_csv(OUTPUT_TABLE_DIR / "02_scaling_parameters.csv", index=False)
model_df.to_csv(OUTPUT_TABLE_DIR / "02_modeling_dataset.csv", index=False)

display(sample_flow)
display(scaling_parameters)
print("Model rows       :", f"{len(model_df):,}")
print("Province clusters:", model_df["provinsi_pt"].nunique())
print("Districts        :", model_df["kabupaten_pt"].nunique())
print("Total graduates  :", f"{model_df['jumlah_lulusan'].sum():,.0f}")
print("Total enrolled   :", f"{model_df['jumlah_terdaftar'].sum():,.0f}")


,stage,n_rows,excluded_from_previous,retained_pct_of_initial
0,Dataset A awal,2624,0,100.000000
1,Complete-case seluruh prediktor utama dan sens...,2567,57,97.827744
2,Exposure positif dan outcome nonnegatif,2567,0,97.827744


,variable,mean,std_population,minimum,maximum
0,active_share,0.831539,0.156051,0.017888,1.000000
1,mbkm_share,0.016803,0.063694,0.000000,0.925795
2,log_pt,2.299541,0.956289,0.693147,4.727388
3,pt_high_share,0.057677,0.112687,0.000000,1.000000
4,prof_share,0.020166,0.028905,0.000000,0.133621
5,cert_share,0.452262,0.171850,0.000000,1.000000
6,senior_share,0.091317,0.088165,0.000000,0.425000


Model rows       : 2,567
Province clusters: 34
Districts        : 396
Total graduates  : 2,544,731
Total enrolled   : 10,715,358


## 3. Diagnostik Poisson dan overdispersion

Model Poisson dengan spesifikasi prediktor utama digunakan sebagai diagnostik,
bukan sebagai model inferensi akhir. Nilai Pearson dispersion yang jauh lebih
besar dari satu menunjukkan bahwa asumsi equidispersion Poisson tidak sesuai.


In [6]:
BASE_CATEGORICAL = (
    'C(jk, Treatment(reference="L"))'
    ' + C(nm_jenj_didik, Treatment(reference="S1"))'
)
MAIN_NUMERIC = (
    "z_active_share + z_mbkm_share + z_log_pt"
    " + z_pt_high_share + z_prof_share + z_cert_share"
)
PRIMARY_FORMULA = "jumlah_lulusan ~ " + BASE_CATEGORICAL + " + " + MAIN_NUMERIC

poisson_result = smf.glm(
    formula=PRIMARY_FORMULA,
    data=model_df,
    family=sm.families.Poisson(),
    offset=np.log(model_df["jumlah_terdaftar"]),
).fit(
    cov_type="cluster",
    cov_kwds={"groups": model_df["provinsi_pt"], "use_correction": True},
)

poisson_diagnostics = pd.DataFrame([{
    "model": "Poisson primary specification",
    "nobs": int(poisson_result.nobs),
    "aic": float(poisson_result.aic),
    "pearson_chi2": float(poisson_result.pearson_chi2),
    "df_resid": float(poisson_result.df_resid),
    "pearson_dispersion": float(poisson_result.pearson_chi2 / poisson_result.df_resid),
    "deviance_per_df": float(poisson_result.deviance / poisson_result.df_resid),
}])
poisson_diagnostics.to_csv(OUTPUT_TABLE_DIR / "02_poisson_overdispersion_diagnostics.csv", index=False)
display(poisson_diagnostics)
assert poisson_diagnostics.loc[0, "pearson_dispersion"] > 1
print("Pearson dispersion:", f"{poisson_diagnostics.loc[0, 'pearson_dispersion']:,.2f}")


,model,nobs,aic,pearson_chi2,df_resid,pearson_dispersion,deviance_per_df
0,Poisson primary specification,2567,"332,253.441551","475,391.430287","2,549.000000",186.501150,123.611060


Pearson dispersion: 186.50


## 4. Model Negative Binomial bertingkat

Urutan model ditetapkan sebelum interpretasi hasil:

- **M0:** offset saja;
- **M1:** gender dan jenjang;
- **M2:** tambahan input/partisipasi mahasiswa;
- **M3:** tambahan kapasitas dan akreditasi perguruan tinggi;
- **M4:** tambahan kualitas dosen — **model inferensi utama**;
- **M5:** interaksi gender×jenjang — model ekstensi.

Standard error dikelompokkan menurut provinsi dengan finite-cluster correction.


In [7]:
FORMULAS = {
    "M0_offset_only": "jumlah_lulusan ~ 1",
    "M1_gender_level": "jumlah_lulusan ~ " + BASE_CATEGORICAL,
    "M2_student_input": (
        "jumlah_lulusan ~ " + BASE_CATEGORICAL + " + z_active_share + z_mbkm_share"
    ),
    "M3_institutional_capacity": (
        "jumlah_lulusan ~ " + BASE_CATEGORICAL
        + " + z_active_share + z_mbkm_share + z_log_pt + z_pt_high_share"
    ),
    "M4_faculty_quality_primary": PRIMARY_FORMULA,
    "M5_gender_level_interaction": (
        "jumlah_lulusan ~ "
        'C(jk, Treatment(reference="L")) * C(nm_jenj_didik, Treatment(reference="S1"))'
        " + " + MAIN_NUMERIC
    ),
}

nb_results = {}
model_rows = []
for name, formula in FORMULAS.items():
    print(f"Fitting {name} ...")
    result = fit_nb2_clustered(formula, model_df)
    nb_results[name] = result
    model_rows.append(model_comparison_row(name, result))

model_comparison = pd.DataFrame(model_rows).sort_values("aic").reset_index(drop=True)
model_comparison["delta_aic"] = model_comparison["aic"] - model_comparison["aic"].min()
model_comparison["aic_weight"] = np.exp(-0.5 * model_comparison["delta_aic"])
model_comparison["aic_weight"] = model_comparison["aic_weight"] / model_comparison["aic_weight"].sum()
model_comparison.to_csv(OUTPUT_TABLE_DIR / "02_model_comparison.csv", index=False)
display(model_comparison)
assert model_comparison["converged"].all(), "Ada model yang tidak konvergen."


Fitting M0_offset_only ...
Fitting M1_gender_level ...
Fitting M2_student_input ...


Fitting M3_institutional_capacity ...
Fitting M4_faculty_quality_primary ...


Fitting M5_gender_level_interaction ...


,model,nobs,df_model,df_resid,log_likelihood,aic,bic,mcfadden_pseudo_r2,alpha,converged,covariance,delta_aic,aic_weight
0,M4_faculty_quality_primary,2567,17.000000,"2,549.000000","-14,364.316119","28,766.632238","28,877.791609",0.110374,0.207689,True,cluster,0.000000,0.479580
1,M3_institutional_capacity,2567,15.000000,"2,551.000000","-14,366.323629","28,766.647258","28,866.105642",0.110250,0.208180,True,cluster,0.015020,0.475992
2,M5_gender_level_interaction,2567,27.000000,"2,539.000000","-14,357.316531","28,772.633063","28,942.297365",0.110808,0.206540,True,cluster,6.000825,0.023867
3,M2_student_input,2567,13.000000,"2,553.000000","-14,371.465667","28,772.931334","28,860.688732",0.109932,0.209165,True,cluster,6.299096,0.020560
4,M1_gender_level,2567,11.000000,"2,555.000000","-14,505.994154","29,037.988308","29,114.044719",0.101600,0.234556,True,cluster,271.356070,0.000000
5,M0_offset_only,2567,0.000000,"2,566.000000","-16,146.472259","32,296.944519","32,308.645505",0.000000,0.806167,True,cluster,"3,530.312280",0.000000


In [8]:
model_order = list(FORMULAS)
lr_tests = []
for reduced_name, full_name in zip(model_order[:-1], model_order[1:]):
    lr_tests.append(likelihood_ratio_test(
        reduced_name, nb_results[reduced_name], full_name, nb_results[full_name]
    ))
lr_tests = pd.DataFrame(lr_tests)
lr_tests.to_csv(OUTPUT_TABLE_DIR / "02_likelihood_ratio_tests.csv", index=False)
display(lr_tests)

PRIMARY_MODEL_NAME = "M4_faculty_quality_primary"
PRIMARY_RESULT = nb_results[PRIMARY_MODEL_NAME]
interaction_test = lr_tests.loc[lr_tests["full_model"].eq("M5_gender_level_interaction")].iloc[0]

print("Primary model:", PRIMARY_MODEL_NAME)
print("Primary AIC  :", f"{PRIMARY_RESULT.aic:,.3f}")
print("Primary alpha:", f"{PRIMARY_RESULT.params['alpha']:.6f}")
print("M4 -> M5 interaction p:", f"{interaction_test['p_value']:.6f}")


,reduced_model,full_model,lr_statistic,df_difference,p_value,significant_0_05
0,M0_offset_only,M1_gender_level,"3,280.956211",11,0.000000,True
1,M1_gender_level,M2_student_input,269.056974,2,0.000000,True
2,M2_student_input,M3_institutional_capacity,10.284076,2,0.005846,True
3,M3_institutional_capacity,M4_faculty_quality_primary,4.015020,2,0.134323,False
4,M4_faculty_quality_primary,M5_gender_level_interaction,13.999175,10,0.173029,False


Primary model: M4_faculty_quality_primary
Primary AIC  : 28,766.632
Primary alpha: 0.207689
M4 -> M5 interaction p: 0.173029


In [9]:
plot_df = model_comparison.sort_values("aic", ascending=False)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(plot_df["model"], plot_df["aic"])
ax.set_xlabel("Akaike Information Criterion (lebih rendah lebih baik)")
ax.set_ylabel("")
ax.set_title("Perbandingan Model Negative Binomial")
ax.grid(axis="x", alpha=0.25)
for index, row in plot_df.reset_index(drop=True).iterrows():
    ax.text(row["aic"] + 10, index, f"{row['aic']:,.1f}", va="center", fontsize=8)
save_figure(fig, "02_01_aic_model_comparison.png")


PosixPath('/mnt/data/work_project02/pt_education_research_project/outputs/figures/02_01_aic_model_comparison.png')

## 5. IRR model utama

- Prediktor numerik: IRR untuk kenaikan satu standar deviasi.
- Gender: perempuan (`P`) dibandingkan laki-laki (`L`).
- Jenjang: setiap kategori dibandingkan S1.
- Interval kepercayaan dan p-value menggunakan cluster-robust covariance
  berdasarkan provinsi.


In [10]:
irr_primary = extract_irr_table(PRIMARY_RESULT, PRIMARY_MODEL_NAME)
TERM_LABELS = {
    "Intercept": "Intercept",
    'C(jk, Treatment(reference="L"))[T.P]': "Perempuan vs laki-laki",
    "z_active_share": "Proporsi mahasiswa aktif (per 1 SD)",
    "z_mbkm_share": "Proporsi MBKM (per 1 SD)",
    "z_log_pt": "Log jumlah PT (per 1 SD)",
    "z_pt_high_share": "Proporsi PT akreditasi tinggi (per 1 SD)",
    "z_prof_share": "Proporsi profesor (per 1 SD)",
    "z_senior_share": "Proporsi dosen senior (per 1 SD)",
    "z_cert_share": "Proporsi dosen tersertifikasi (per 1 SD)",
    "alpha": "Alpha overdispersion",
}

def readable_term(term: str) -> str:
    if term in TERM_LABELS:
        return TERM_LABELS[term]
    prefix = 'C(nm_jenj_didik, Treatment(reference="S1"))[T.'
    if term.startswith(prefix) and term.endswith("]"):
        return f"{term[len(prefix):-1]} vs S1"
    return term

irr_primary["term_label"] = irr_primary["term"].map(readable_term)
irr_primary["term_type"] = np.select(
    [
        irr_primary["term"].eq("Intercept"),
        irr_primary["term"].eq("alpha"),
        irr_primary["term"].str.startswith("C(jk"),
        irr_primary["term"].str.startswith("C(nm_jenj"),
        irr_primary["term"].str.startswith("z_"),
    ],
    ["intercept", "dispersion", "gender", "education_level", "continuous_standardized"],
    default="other",
)
irr_primary.to_csv(OUTPUT_TABLE_DIR / "02_primary_model_irr.csv", index=False)
display(irr_primary.loc[~irr_primary["term"].isin(["Intercept", "alpha"]), [
    "term_label", "coefficient", "std_error", "irr", "irr_ci_lower", "irr_ci_upper", "p_value"
]])
alpha_row = irr_primary.loc[irr_primary["term"].eq("alpha")].iloc[0]
print("Alpha NB2:", f"{alpha_row['coefficient']:.6f}", "| p:", f"{alpha_row['p_value']:.3e}")


,term_label,coefficient,std_error,irr,irr_ci_lower,irr_ci_upper,p_value
1,Perempuan vs laki-laki,0.090051,0.013334,1.094230,1.066003,1.123204,0.000000
2,D1 vs S1,1.528229,0.091717,4.610003,3.851512,5.517867,0.000000
3,D2 vs S1,0.871745,0.065799,2.391080,2.101774,2.720207,0.000000
4,D3 vs S1,0.487844,0.025271,1.628801,1.550092,1.711506,0.000000
5,D4 vs S1,0.023833,0.044164,1.024119,0.939200,1.116717,0.589440
6,Profesi vs S1,2.052996,0.102629,7.791209,6.371571,9.527154,0.000000
7,S2 vs S1,0.677653,0.032851,1.969250,1.846452,2.100214,0.000000
8,S2 Terapan vs S1,0.791566,0.095831,2.206850,1.828947,2.662836,0.000000
9,S3 vs S1,-0.051197,0.051381,0.950092,0.859073,1.050754,0.319047
10,Sp-1 vs S1,0.014659,0.073243,1.014767,0.879066,1.171416,0.841370


Alpha NB2: 0.207689 | p: 2.080e-13


In [11]:
forest = irr_primary.loc[~irr_primary["term"].isin(["Intercept", "alpha"])].copy().sort_values("irr")
fig_height = max(6, 0.42 * len(forest))
fig, ax = plt.subplots(figsize=(9, fig_height))
y_positions = np.arange(len(forest))
xerr = np.vstack([
    forest["irr"] - forest["irr_ci_lower"],
    forest["irr_ci_upper"] - forest["irr"],
])
ax.errorbar(forest["irr"], y_positions, xerr=xerr, fmt="o", capsize=3)
ax.axvline(1.0, linestyle="--", linewidth=1)
ax.set_yticks(y_positions)
ax.set_yticklabels(forest["term_label"])
ax.set_xscale("log")
ax.set_xlabel("Incidence Rate Ratio (skala log)")
ax.set_title("IRR Model Negative Binomial Utama")
ax.grid(axis="x", alpha=0.25)
save_figure(fig, "02_02_primary_model_irr.png")


PosixPath('/mnt/data/work_project02/pt_education_research_project/outputs/figures/02_02_primary_model_irr.png')

## 6. Multikolinearitas prediktor numerik

VIF dihitung pada prediktor numerik terstandardisasi. Kategori gender dan
jenjang tidak dimasukkan ke tabel VIF sederhana ini karena memerlukan
interpretasi generalized VIF yang berbeda.


In [12]:
VIF_COLUMNS = [
    "z_active_share", "z_mbkm_share", "z_log_pt", "z_pt_high_share", "z_prof_share", "z_cert_share"
]
vif_matrix = sm.add_constant(model_df[VIF_COLUMNS])
vif_table = pd.DataFrame({
    "term": vif_matrix.columns,
    "vif": [variance_inflation_factor(vif_matrix.to_numpy(), index) for index in range(vif_matrix.shape[1])],
})
vif_table.to_csv(OUTPUT_TABLE_DIR / "02_numeric_predictor_vif.csv", index=False)
display(vif_table)
assert vif_table.loc[vif_table["term"].ne("const"), "vif"].max() < 10


,term,vif
0,const,1.000000
1,z_active_share,1.032437
2,z_mbkm_share,1.021557
3,z_log_pt,1.510605
4,z_pt_high_share,1.496514
5,z_prof_share,2.058314
6,z_cert_share,1.322622


## 7. Performa in-sample, kalibrasi, dan residual

Ukuran ini digunakan untuk diagnostik spesifikasi, **bukan** sebagai estimasi
generalisasi ke provinsi baru. Validasi out-of-province akan dilakukan secara
khusus pada Notebook 03.


In [13]:
model_df["predicted_primary"] = PRIMARY_RESULT.predict(
    model_df, offset=np.log(model_df["jumlah_terdaftar"])
)
alpha_primary = float(PRIMARY_RESULT.params["alpha"])
model_df["raw_residual"] = model_df["jumlah_lulusan"] - model_df["predicted_primary"]
model_df["pearson_residual"] = model_df["raw_residual"] / np.sqrt(
    model_df["predicted_primary"] + alpha_primary * model_df["predicted_primary"] ** 2
)
model_df["prediction_ratio"] = safe_divide(model_df["jumlah_lulusan"], model_df["predicted_primary"])

performance_primary = pd.DataFrame([{
    "model": PRIMARY_MODEL_NAME,
    **prediction_metrics(model_df["jumlah_lulusan"], model_df["predicted_primary"], alpha_primary),
    "mcfadden_pseudo_r2": float(PRIMARY_RESULT.prsquared),
}])
performance_primary.to_csv(OUTPUT_TABLE_DIR / "02_primary_in_sample_performance.csv", index=False)

calibration = calibration_by_quantile(model_df, "jumlah_lulusan", "predicted_primary", q=10)
calibration.to_csv(OUTPUT_TABLE_DIR / "02_primary_calibration_deciles.csv", index=False)

residual_summary = pd.DataFrame([{
    "n": len(model_df),
    "pearson_mean": float(model_df["pearson_residual"].mean()),
    "pearson_std": float(model_df["pearson_residual"].std(ddof=1)),
    "pearson_min": float(model_df["pearson_residual"].min()),
    "pearson_max": float(model_df["pearson_residual"].max()),
    "abs_residual_gt_2_n": int(model_df["pearson_residual"].abs().gt(2).sum()),
    "abs_residual_gt_2_pct": float(model_df["pearson_residual"].abs().gt(2).mean() * 100),
    "abs_residual_gt_3_n": int(model_df["pearson_residual"].abs().gt(3).sum()),
    "abs_residual_gt_3_pct": float(model_df["pearson_residual"].abs().gt(3).mean() * 100),
    "spearman_fitted_abs_residual": float(stats.spearmanr(
        model_df["predicted_primary"], model_df["pearson_residual"].abs(), nan_policy="omit"
    ).statistic),
}])
residual_summary.to_csv(OUTPUT_TABLE_DIR / "02_residual_summary.csv", index=False)

display(performance_primary)
display(calibration)
display(residual_summary)


,model,mae,rmse,rmsle,mean_poisson_deviance,mean_nb_deviance,observed_total,predicted_total,predicted_to_observed_ratio,mcfadden_pseudo_r2
0,M4_faculty_quality_primary,254.726909,"1,640.850426",0.528685,136.843330,1.068692,"2,544,731.000000","2,765,596.524285",1.086793,0.110374


,prediction_decile,n,observed_total,predicted_total,observed_mean,predicted_mean,observed_to_predicted_ratio
0,1,257,1588,"1,409.774262",6.178988,5.485503,1.126421
1,2,257,4178,"4,281.715252",16.256809,16.660371,0.975777
2,3,256,8607,"8,793.695990",33.621094,34.350375,0.978769
3,4,257,14854,"15,885.085403",57.797665,61.809671,0.935091
4,5,257,27520,"26,076.800492",107.081712,101.466150,1.055344
5,6,256,43902,"44,107.950725",171.492188,172.296683,0.995331
6,7,257,80729,"78,554.320239",314.120623,305.658834,1.027684
7,8,256,150991,"147,714.211428",589.808594,577.008638,1.022183
8,9,257,335683,"342,054.891589","1,306.159533","1,330.952886",0.981372
9,10,257,1876679,"2,096,718.078906","7,302.252918","8,158.436105",0.895055


,n,pearson_mean,pearson_std,pearson_min,pearson_max,abs_residual_gt_2_n,abs_residual_gt_2_pct,abs_residual_gt_3_n,abs_residual_gt_3_pct,spearman_fitted_abs_residual
0,2567,0.015192,1.369524,-2.166646,28.764070,98,3.817686,45,1.753019,-0.228092


In [14]:
outlier_columns = [
    "provinsi_pt", "kabupaten_pt", "jk", "nm_jenj_didik",
    "jumlah_lulusan", "jumlah_terdaftar", "predicted_primary",
    "raw_residual", "pearson_residual", "prediction_ratio",
]
largest_positive = model_df.nlargest(20, "pearson_residual")[outlier_columns].assign(residual_direction="positive")
largest_negative = model_df.nsmallest(20, "pearson_residual")[outlier_columns].assign(residual_direction="negative")
residual_outliers = pd.concat([largest_positive, largest_negative], ignore_index=True)
residual_outliers["small_denominator_lt_100"] = residual_outliers["jumlah_terdaftar"] < 100
residual_outliers.to_csv(OUTPUT_TABLE_DIR / "02_residual_outliers_top20_each_direction.csv", index=False)

province_residuals = model_df.groupby("provinsi_pt", as_index=False).agg(
    observed_graduates=("jumlah_lulusan", "sum"),
    predicted_graduates=("predicted_primary", "sum"),
    registered_students=("jumlah_terdaftar", "sum"),
    n_strata=("jumlah_lulusan", "size"),
)
province_residuals["raw_residual"] = province_residuals["observed_graduates"] - province_residuals["predicted_graduates"]
province_residuals["observed_to_predicted_ratio"] = safe_divide(
    province_residuals["observed_graduates"], province_residuals["predicted_graduates"]
)
province_residuals.to_csv(OUTPUT_TABLE_DIR / "02_province_residuals.csv", index=False)
display(residual_outliers.head(10))


,provinsi_pt,kabupaten_pt,jk,nm_jenj_didik,jumlah_lulusan,jumlah_terdaftar,predicted_primary,raw_residual,pearson_residual,prediction_ratio,residual_direction,small_denominator_lt_100
0,Prov. Maluku Utara,Kota Ternate,P,Profesi,3871,172.000000,272.144134,"3,598.855866",28.764070,14.224080,positive,False
1,Prov. Sulawesi Utara,Kab. Minahasa,L,Profesi,1416,75.000000,122.672544,"1,293.327456",22.693122,11.542925,positive,True
2,Prov. Sumatera Utara,Kab. Batubara,P,S1,15,2.000000,0.417491,14.582509,21.649713,35.928910,positive,True
3,Prov. Sulawesi Utara,Kab. Minahasa,P,Profesi,5848,312.000000,558.405025,"5,289.594975",20.696737,10.472685,positive,False
4,Prov. Maluku Utara,Kota Ternate,L,Profesi,924,82.000000,112.769310,811.230690,15.458507,8.193719,positive,True
5,Prov. Sumatera Selatan,Kab. Ogan Ilir,L,Profesi,2805,499.000000,656.883903,"2,148.116097",7.149512,4.270161,positive,False
6,Prov. Aceh,Kab. Aceh Tengah,P,Profesi,1010,146.000000,240.021853,769.978147,6.969600,4.207950,positive,False
7,Prov. Bengkulu,Kab. Rejang Lebong,L,Profesi,717,196.000000,183.580928,533.419072,6.293790,3.905634,positive,False
8,Prov. Jambi,Kab. Kerinci,L,Profesi,519,86.000000,132.931744,386.068256,6.260404,3.904259,positive,True
9,Prov. Kalimantan Selatan,Kab. Tanah Bumbu,L,S1,36,54.000000,8.701615,27.298385,5.523291,4.137163,positive,True


In [15]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(model_df["predicted_primary"], model_df["jumlah_lulusan"], alpha=0.35, s=18)
minimum = min(model_df["predicted_primary"].min(), model_df["jumlah_lulusan"].min())
maximum = max(model_df["predicted_primary"].max(), model_df["jumlah_lulusan"].max())
ax.plot([minimum, maximum], [minimum, maximum], linestyle="--")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Prediksi jumlah lulusan")
ax.set_ylabel("Jumlah lulusan observasi")
ax.set_title("Observed vs Predicted — Model Negative Binomial Utama")
ax.grid(alpha=0.2)
save_figure(fig, "02_03_observed_vs_predicted_log.png")

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.scatter(model_df["predicted_primary"], model_df["pearson_residual"], alpha=0.35, s=18)
ax.axhline(0, linestyle="--", linewidth=1)
ax.axhline(2, linestyle=":", linewidth=1)
ax.axhline(-2, linestyle=":", linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("Prediksi jumlah lulusan (skala log)")
ax.set_ylabel("Pearson residual")
ax.set_title("Pearson Residual vs Fitted")
ax.grid(alpha=0.2)
save_figure(fig, "02_04_pearson_residual_vs_fitted.png")

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.hist(model_df["pearson_residual"], bins=50)
ax.axvline(0, linestyle="--", linewidth=1)
ax.set_xlabel("Pearson residual")
ax.set_ylabel("Frekuensi")
ax.set_title("Distribusi Pearson Residual")
save_figure(fig, "02_05_pearson_residual_distribution.png")

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot(calibration["predicted_mean"], calibration["observed_mean"], marker="o")
cal_min = min(calibration["predicted_mean"].min(), calibration["observed_mean"].min())
cal_max = max(calibration["predicted_mean"].max(), calibration["observed_mean"].max())
ax.plot([cal_min, cal_max], [cal_min, cal_max], linestyle="--")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Rata-rata prediksi per desil")
ax.set_ylabel("Rata-rata observasi per desil")
ax.set_title("Kalibrasi In-Sample berdasarkan Desil Prediksi")
ax.grid(alpha=0.2)
save_figure(fig, "02_06_calibration_deciles.png")


PosixPath('/mnt/data/work_project02/pt_education_research_project/outputs/figures/02_06_calibration_deciles.png')

## 8. Kecocokan menurut gender dan jenjang

Prediksi dijumlahkan pada setiap kelompok gender×jenjang, kemudian dibandingkan
dengan total observasi. Penggabungan ini mempertahankan exposure aktual setiap
strata.


In [16]:
group_fit = model_df.groupby(["nm_jenj_didik", "jk"], as_index=False).agg(
    observed_graduates=("jumlah_lulusan", "sum"),
    predicted_graduates=("predicted_primary", "sum"),
    registered_students=("jumlah_terdaftar", "sum"),
    n_strata=("jumlah_lulusan", "size"),
)
group_fit["observed_per_1000_registered"] = safe_divide(
    group_fit["observed_graduates"], group_fit["registered_students"]
) * 1000
group_fit["predicted_per_1000_registered"] = safe_divide(
    group_fit["predicted_graduates"], group_fit["registered_students"]
) * 1000
group_fit["observed_to_predicted_ratio"] = safe_divide(
    group_fit["observed_graduates"], group_fit["predicted_graduates"]
)
group_fit.to_csv(OUTPUT_TABLE_DIR / "02_fit_by_gender_and_level.csv", index=False)
display(group_fit)

levels = list(model_df.groupby("nm_jenj_didik")["jumlah_lulusan"].sum().sort_values(ascending=False).index)
x = np.arange(len(levels))
width = 0.19
fig, ax = plt.subplots(figsize=(13, 6))
for index, gender in enumerate(["L", "P"]):
    subset = group_fit.loc[group_fit["jk"].eq(gender)].set_index("nm_jenj_didik").reindex(levels)
    offset = (index - 0.5) * 2 * width
    ax.bar(x + offset, subset["observed_per_1000_registered"], width=width, label=f"Observasi {gender}")
    ax.bar(x + offset + width, subset["predicted_per_1000_registered"], width=width, label=f"Prediksi {gender}", alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(levels, rotation=45, ha="right")
ax.set_ylabel("Lulusan per 1.000 mahasiswa terdaftar")
ax.set_title("Kecocokan Model menurut Gender dan Jenjang")
ax.legend(ncol=2)
ax.grid(axis="y", alpha=0.2)
save_figure(fig, "02_07_fit_by_gender_and_level.png")


,nm_jenj_didik,jk,observed_graduates,predicted_graduates,registered_students,n_strata,observed_per_1000_registered,predicted_per_1000_registered,observed_to_predicted_ratio
0,D1,L,1355,988.018481,"1,477.000000",14,917.400135,668.936006,1.371432
1,D1,P,438,471.593531,549.000000,13,797.814208,859.004611,0.928766
2,D2,L,517,581.321054,"1,455.000000",24,355.326460,399.533370,0.889354
3,D2,P,441,510.661376,"1,177.000000",21,374.681393,433.866930,0.863586
4,D3,L,48950,"53,428.627056","210,798.000000",247,232.212829,253.458890,0.916176
5,D3,P,91658,"94,108.968395","318,087.000000",276,288.153870,295.859210,0.973956
6,D4,L,29964,"32,211.333601","179,931.000000",119,166.530503,179.020478,0.930232
7,D4,P,45080,"43,168.047039","212,506.000000",116,212.135187,203.138015,1.044291
8,Profesi,L,248193,"267,495.087789","211,851.000000",171,"1,171.545095","1,262.656715",0.927841
9,Profesi,P,668260,"787,208.094385","560,418.000000",184,"1,192.431364","1,404.680247",0.848899


PosixPath('/mnt/data/work_project02/pt_education_research_project/outputs/figures/02_07_fit_by_gender_and_level.png')

In [17]:
province_plot = province_residuals.sort_values("raw_residual")
fig_height = max(7, 0.25 * len(province_plot))
fig, ax = plt.subplots(figsize=(9, fig_height))
ax.barh(province_plot["provinsi_pt"], province_plot["raw_residual"])
ax.axvline(0, linestyle="--", linewidth=1)
ax.set_xlabel("Observed minus predicted graduates")
ax.set_ylabel("")
ax.set_title("Residual Agregat menurut Provinsi")
ax.grid(axis="x", alpha=0.2)
save_figure(fig, "02_08_province_aggregate_residuals.png")


PosixPath('/mnt/data/work_project02/pt_education_research_project/outputs/figures/02_08_province_aggregate_residuals.png')

## 9. Analisis sensitivitas

Analisis yang telah ditetapkan:

1. **Small denominator:** mengecualikan strata dengan mahasiswa terdaftar <100.
2. **Faculty specification:** mengganti proporsi profesor dengan proporsi dosen senior.
3. **Covariance:** menggunakan HC1 alih-alih cluster provinsi.
4. **Province fixed effects:** menambahkan efek tetap provinsi.
5. **Interaction extension:** model M5 dari rangkaian utama.

Prediktor standar pada subset denominator tetap menggunakan parameter
standardisasi sampel utama sehingga koefisien dapat dibandingkan.


In [18]:
SENSITIVITY_FORMULAS = {
    "S1_exclude_enrolled_lt_100": PRIMARY_FORMULA,
    "S2_replace_professor_with_senior": PRIMARY_FORMULA.replace("z_prof_share", "z_senior_share"),
    "S3_hc1_covariance": PRIMARY_FORMULA,
    "S4_province_fixed_effects": PRIMARY_FORMULA + " + C(provinsi_pt)",
    "S5_gender_level_interaction": FORMULAS["M5_gender_level_interaction"],
}

sensitivity_results = {}
sensitivity_results["S1_exclude_enrolled_lt_100"] = fit_nb2_clustered(
    SENSITIVITY_FORMULAS["S1_exclude_enrolled_lt_100"],
    model_df.loc[model_df["jumlah_terdaftar"] >= 100].copy(),
)
sensitivity_results["S2_replace_professor_with_senior"] = fit_nb2_clustered(
    SENSITIVITY_FORMULAS["S2_replace_professor_with_senior"], model_df
)
sensitivity_results["S3_hc1_covariance"] = fit_nb2_hc1(
    SENSITIVITY_FORMULAS["S3_hc1_covariance"], model_df
)
sensitivity_results["S4_province_fixed_effects"] = fit_nb2_clustered(
    SENSITIVITY_FORMULAS["S4_province_fixed_effects"], model_df
)
sensitivity_results["S5_gender_level_interaction"] = nb_results["M5_gender_level_interaction"]

sensitivity_model_rows = [model_comparison_row(PRIMARY_MODEL_NAME, PRIMARY_RESULT)]
for name, result in sensitivity_results.items():
    sensitivity_model_rows.append(model_comparison_row(name, result))
sensitivity_model_summary = pd.DataFrame(sensitivity_model_rows)
sensitivity_model_summary.to_csv(OUTPUT_TABLE_DIR / "02_sensitivity_model_summary.csv", index=False)

sensitivity_terms = [
    'C(jk, Treatment(reference="L"))[T.P]', "z_active_share", "z_mbkm_share",
    "z_log_pt", "z_pt_high_share", "z_prof_share", "z_senior_share", "z_cert_share",
]
sensitivity_coefficients = []
all_sensitivity_models = {PRIMARY_MODEL_NAME: PRIMARY_RESULT, **sensitivity_results}
for model_name, result in all_sensitivity_models.items():
    table = extract_irr_table(result, model_name)
    table = table.loc[table["term"].isin(sensitivity_terms)].copy()
    table["term_label"] = table["term"].map(readable_term)
    sensitivity_coefficients.append(table)
sensitivity_coefficients = pd.concat(sensitivity_coefficients, ignore_index=True)
sensitivity_coefficients.to_csv(OUTPUT_TABLE_DIR / "02_sensitivity_selected_irr.csv", index=False)

display(sensitivity_model_summary)
display(sensitivity_coefficients[["model", "term_label", "irr", "irr_ci_lower", "irr_ci_upper", "p_value"]])


,model,nobs,df_model,df_resid,log_likelihood,aic,bic,mcfadden_pseudo_r2,alpha,converged,covariance
0,M4_faculty_quality_primary,2567,17.000000,"2,549.000000","-14,364.316119","28,766.632238","28,877.791609",0.110374,0.207689,True,cluster
1,S1_exclude_enrolled_lt_100,1980,17.000000,"1,962.000000","-12,324.041759","24,686.083517","24,792.309708",0.116046,0.181276,True,cluster
2,S2_replace_professor_with_senior,2567,17.000000,"2,549.000000","-14,363.191747","28,764.383494","28,875.542864",0.110444,0.207472,True,cluster
3,S3_hc1_covariance,2567,17.000000,"2,549.000000","-14,364.316119","28,766.632238","28,877.791609",0.110374,0.207689,True,HC1
4,S4_province_fixed_effects,2567,50.000000,"2,516.000000","-14,265.791219","28,635.582437","28,939.808083",0.116476,0.190197,True,cluster
5,S5_gender_level_interaction,2567,27.000000,"2,539.000000","-14,357.316531","28,772.633063","28,942.297365",0.110808,0.206540,True,cluster


,model,term_label,irr,irr_ci_lower,irr_ci_upper,p_value
0,M4_faculty_quality_primary,Perempuan vs laki-laki,1.094230,1.066003,1.123204,0.000000
1,M4_faculty_quality_primary,Proporsi mahasiswa aktif (per 1 SD),1.194223,1.135958,1.255476,0.000000
2,M4_faculty_quality_primary,Proporsi MBKM (per 1 SD),0.990447,0.968323,1.013077,0.404963
3,M4_faculty_quality_primary,Log jumlah PT (per 1 SD),0.978508,0.947174,1.010880,0.190766
4,M4_faculty_quality_primary,Proporsi PT akreditasi tinggi (per 1 SD),1.018924,0.961838,1.079398,0.523934
5,M4_faculty_quality_primary,Proporsi profesor (per 1 SD),1.021741,0.973139,1.072770,0.387069
6,M4_faculty_quality_primary,Proporsi dosen tersertifikasi (per 1 SD),1.010805,0.979855,1.042732,0.498193
7,S1_exclude_enrolled_lt_100,Perempuan vs laki-laki,1.129547,1.094002,1.166248,0.000000
8,S1_exclude_enrolled_lt_100,Proporsi mahasiswa aktif (per 1 SD),1.192298,1.128853,1.259309,0.000000
9,S1_exclude_enrolled_lt_100,Proporsi MBKM (per 1 SD),0.991987,0.969410,1.015089,0.493369


In [19]:
plot_terms = [
    "Perempuan vs laki-laki", "Proporsi mahasiswa aktif (per 1 SD)",
    "Proporsi MBKM (per 1 SD)", "Log jumlah PT (per 1 SD)",
    "Proporsi PT akreditasi tinggi (per 1 SD)", "Proporsi profesor (per 1 SD)",
    "Proporsi dosen senior (per 1 SD)", "Proporsi dosen tersertifikasi (per 1 SD)",
]
sens_plot = sensitivity_coefficients.loc[sensitivity_coefficients["term_label"].isin(plot_terms)].copy()
models_for_plot = [
    PRIMARY_MODEL_NAME, "S1_exclude_enrolled_lt_100",
    "S2_replace_professor_with_senior", "S4_province_fixed_effects",
]
sens_plot = sens_plot.loc[sens_plot["model"].isin(models_for_plot)].copy()
fig, ax = plt.subplots(figsize=(10, 8))
term_positions = {term: position for position, term in enumerate(plot_terms)}
model_offsets = {model: offset for model, offset in zip(models_for_plot, np.linspace(-0.24, 0.24, len(models_for_plot)))}
for model_name in models_for_plot:
    subset = sens_plot.loc[sens_plot["model"].eq(model_name)]
    if subset.empty:
        continue
    y = np.array([term_positions[label] for label in subset["term_label"]]) + model_offsets[model_name]
    xerr = np.vstack([
        subset["irr"] - subset["irr_ci_lower"],
        subset["irr_ci_upper"] - subset["irr"],
    ])
    ax.errorbar(subset["irr"], y, xerr=xerr, fmt="o", capsize=2, label=model_name)
ax.axvline(1.0, linestyle="--", linewidth=1)
ax.set_yticks(range(len(plot_terms)))
ax.set_yticklabels(plot_terms)
ax.set_xscale("log")
ax.set_xlabel("IRR (skala log)")
ax.set_title("Stabilitas IRR pada Analisis Sensitivitas")
ax.legend(fontsize=8)
ax.grid(axis="x", alpha=0.2)
save_figure(fig, "02_09_sensitivity_irr.png")


PosixPath('/mnt/data/work_project02/pt_education_research_project/outputs/figures/02_09_sensitivity_irr.png')

## 10. Penyimpanan model dan ringkasan siap artikel


In [20]:
PRIMARY_RESULT.save(OUTPUT_MODEL_DIR / "02_nb2_primary_m4.pkl")
nb_results["M5_gender_level_interaction"].save(OUTPUT_MODEL_DIR / "02_nb2_interaction_m5.pkl")
sensitivity_results["S4_province_fixed_effects"].save(OUTPUT_MODEL_DIR / "02_nb2_province_fixed_effects.pkl")

with (OUTPUT_TABLE_DIR / "02_primary_model_summary.txt").open("w", encoding="utf-8") as handle:
    handle.write(PRIMARY_RESULT.summary().as_text())

female_term = irr_primary.loc[
    irr_primary["term"].eq('C(jk, Treatment(reference="L"))[T.P]')
].iloc[0]
active_term = irr_primary.loc[irr_primary["term"].eq("z_active_share")].iloc[0]
interaction_p = float(interaction_test["p_value"])
pearson_dispersion = float(poisson_diagnostics.loc[0, "pearson_dispersion"])

key_summary = {
    "data_source": DATA_SOURCE,
    "n_model_rows": int(len(model_df)),
    "n_provinces": int(model_df["provinsi_pt"].nunique()),
    "n_districts": int(model_df["kabupaten_pt"].nunique()),
    "primary_model": PRIMARY_MODEL_NAME,
    "primary_aic": float(PRIMARY_RESULT.aic),
    "primary_alpha": float(PRIMARY_RESULT.params["alpha"]),
    "poisson_pearson_dispersion": pearson_dispersion,
    "female_vs_male_irr": float(female_term["irr"]),
    "female_vs_male_ci_lower": float(female_term["irr_ci_lower"]),
    "female_vs_male_ci_upper": float(female_term["irr_ci_upper"]),
    "female_vs_male_p": float(female_term["p_value"]),
    "active_share_irr_per_sd": float(active_term["irr"]),
    "active_share_ci_lower": float(active_term["irr_ci_lower"]),
    "active_share_ci_upper": float(active_term["irr_ci_upper"]),
    "active_share_p": float(active_term["p_value"]),
    "interaction_m4_m5_p": interaction_p,
    "primary_rmsle_in_sample": float(performance_primary.loc[0, "rmsle"]),
    "primary_predicted_to_observed_ratio": float(performance_primary.loc[0, "predicted_to_observed_ratio"]),
    "residual_abs_gt_3_pct": float(residual_summary.loc[0, "abs_residual_gt_3_pct"]),
}
with (OUTPUT_TABLE_DIR / "02_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(key_summary, handle, ensure_ascii=False, indent=2)

key_findings_md = f"""# Ringkasan Hasil Notebook 02

- Sampel model terdiri atas **{len(model_df):,} strata** dari
  **{model_df['provinsi_pt'].nunique()} provinsi** dan
  **{model_df['kabupaten_pt'].nunique()} kabupaten/kota**.
- Poisson menunjukkan Pearson dispersion sebesar
  **{pearson_dispersion:,.2f}**, sehingga asumsi equidispersion tidak layak.
- Model utama M4 mengestimasi alpha NB2 sebesar
  **{PRIMARY_RESULT.params['alpha']:.4f}** dan AIC
  **{PRIMARY_RESULT.aic:,.2f}**.
- Dengan exposure mahasiswa terdaftar dan kovariat lain tetap, strata
  perempuan memiliki IRR **{female_term['irr']:.3f}**
  (95% CI {female_term['irr_ci_lower']:.3f}–{female_term['irr_ci_upper']:.3f},
  p={female_term['p_value']:.3g}) dibandingkan strata laki-laki.
- Kenaikan satu standar deviasi proporsi mahasiswa aktif berhubungan dengan
  IRR **{active_term['irr']:.3f}**
  (95% CI {active_term['irr_ci_lower']:.3f}–{active_term['irr_ci_upper']:.3f},
  p={active_term['p_value']:.3g}).
- Penambahan interaksi gender×jenjang pada M5 menghasilkan p LR
  **{interaction_p:.4f}**; M4 tetap digunakan sebagai model inferensi utama
  karena telah ditetapkan a priori dan lebih parsimonious.
- Seluruh hasil bersifat ekologis dan tidak boleh ditafsirkan sebagai
  probabilitas individual atau efek kausal.
""".strip()
with (OUTPUT_TABLE_DIR / "02_key_findings.md").open("w", encoding="utf-8") as handle:
    handle.write(key_findings_md)
print(key_findings_md)


# Ringkasan Hasil Notebook 02

- Sampel model terdiri atas **2,567 strata** dari
  **34 provinsi** dan
  **396 kabupaten/kota**.
- Poisson menunjukkan Pearson dispersion sebesar
  **186.50**, sehingga asumsi equidispersion tidak layak.
- Model utama M4 mengestimasi alpha NB2 sebesar
  **0.2077** dan AIC
  **28,766.63**.
- Dengan exposure mahasiswa terdaftar dan kovariat lain tetap, strata
  perempuan memiliki IRR **1.094**
  (95% CI 1.066–1.123,
  p=1.44e-11) dibandingkan strata laki-laki.
- Kenaikan satu standar deviasi proporsi mahasiswa aktif berhubungan dengan
  IRR **1.194**
  (95% CI 1.136–1.255,
  p=3.52e-12).
- Penambahan interaksi gender×jenjang pada M5 menghasilkan p LR
  **0.1730**; M4 tetap digunakan sebagai model inferensi utama
  karena telah ditetapkan a priori dan lebih parsimonious.
- Seluruh hasil bersifat ekologis dan tidak boleh ditafsirkan sebagai
  probabilitas individual atau efek kausal.


## 11. Checklist kelayakan Notebook 02


In [21]:
expected_tables = [
    "02_sample_flow.csv", "02_scaling_parameters.csv", "02_modeling_dataset.csv",
    "02_poisson_overdispersion_diagnostics.csv", "02_model_comparison.csv",
    "02_likelihood_ratio_tests.csv", "02_primary_model_irr.csv",
    "02_numeric_predictor_vif.csv", "02_primary_in_sample_performance.csv",
    "02_primary_calibration_deciles.csv", "02_residual_summary.csv",
    "02_residual_outliers_top20_each_direction.csv", "02_province_residuals.csv",
    "02_fit_by_gender_and_level.csv", "02_sensitivity_model_summary.csv",
    "02_sensitivity_selected_irr.csv", "02_summary.json", "02_key_findings.md",
]
expected_figures = [
    "02_01_aic_model_comparison.png", "02_02_primary_model_irr.png",
    "02_03_observed_vs_predicted_log.png", "02_04_pearson_residual_vs_fitted.png",
    "02_05_pearson_residual_distribution.png", "02_06_calibration_deciles.png",
    "02_07_fit_by_gender_and_level.png", "02_08_province_aggregate_residuals.png",
    "02_09_sensitivity_irr.png",
]
expected_models = [
    "02_nb2_primary_m4.pkl", "02_nb2_interaction_m5.pkl", "02_nb2_province_fixed_effects.pkl",
]

readiness_02 = pd.DataFrame([
    {
        "check": "Sampel analitik memadai",
        "status": "PASS" if len(model_df) >= 2000 else "FAIL",
        "evidence": f"{len(model_df):,} strata",
    },
    {
        "check": "Kelompok validasi provinsi tersedia",
        "status": "PASS" if model_df["provinsi_pt"].nunique() >= 30 else "FAIL",
        "evidence": f"{model_df['provinsi_pt'].nunique()} provinsi",
    },
    {
        "check": "Poisson overdispersed",
        "status": "PASS" if pearson_dispersion > 1 else "FAIL",
        "evidence": f"Pearson dispersion={pearson_dispersion:.2f}",
    },
    {
        "check": "Seluruh model utama konvergen",
        "status": "PASS" if model_comparison["converged"].all() else "FAIL",
        "evidence": f"{int(model_comparison['converged'].sum())}/{len(model_comparison)} model",
    },
    {
        "check": "Alpha Negative Binomial positif",
        "status": "PASS" if PRIMARY_RESULT.params["alpha"] > 0 else "FAIL",
        "evidence": f"alpha={PRIMARY_RESULT.params['alpha']:.6f}",
    },
    {
        "check": "VIF prediktor numerik dapat diterima",
        "status": "PASS" if vif_table.loc[vif_table["term"].ne("const"), "vif"].max() < 10 else "FAIL",
        "evidence": f"max VIF={vif_table.loc[vif_table['term'].ne('const'), 'vif'].max():.3f}",
    },
    {
        "check": "Tabel output lengkap",
        "status": "PASS" if all((OUTPUT_TABLE_DIR / file).exists() for file in expected_tables) else "FAIL",
        "evidence": f"{sum((OUTPUT_TABLE_DIR / file).exists() for file in expected_tables)}/{len(expected_tables)} tabel",
    },
    {
        "check": "Figur output lengkap",
        "status": "PASS" if all((OUTPUT_FIGURE_DIR / file).exists() for file in expected_figures) else "FAIL",
        "evidence": f"{sum((OUTPUT_FIGURE_DIR / file).exists() for file in expected_figures)}/{len(expected_figures)} figur",
    },
    {
        "check": "Model tersimpan",
        "status": "PASS" if all((OUTPUT_MODEL_DIR / file).exists() for file in expected_models) else "FAIL",
        "evidence": f"{sum((OUTPUT_MODEL_DIR / file).exists() for file in expected_models)}/{len(expected_models)} model",
    },
])
readiness_02.to_csv(OUTPUT_TABLE_DIR / "02_readiness_checklist.csv", index=False)
display(readiness_02)
assert readiness_02["status"].eq("PASS").all(), "Notebook 02 belum lolos seluruh pemeriksaan."

manifest_rows=[]
for folder, artifact_type in [(OUTPUT_TABLE_DIR,"table"),(OUTPUT_FIGURE_DIR,"figure"),(OUTPUT_MODEL_DIR,"model")]:
    for path in sorted(folder.glob("02_*")):
        if path.is_file():
            manifest_rows.append({
                "artifact_type": artifact_type,
                "filename": path.name,
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "size_bytes": path.stat().st_size,
            })
output_manifest = pd.DataFrame(manifest_rows)
output_manifest.to_csv(OUTPUT_TABLE_DIR / "02_output_manifest.csv", index=False)

print("=" * 76)
print("NOTEBOOK 02 LOLOS.")
print("Model siap dilanjutkan ke 03_Predictive_Modeling_GroupCV.ipynb")
print("=" * 76)


,check,status,evidence
0,Sampel analitik memadai,PASS,"2,567 strata"
1,Kelompok validasi provinsi tersedia,PASS,34 provinsi
2,Poisson overdispersed,PASS,Pearson dispersion=186.50
3,Seluruh model utama konvergen,PASS,6/6 model
4,Alpha Negative Binomial positif,PASS,alpha=0.207689
5,VIF prediktor numerik dapat diterima,PASS,max VIF=2.058
6,Tabel output lengkap,PASS,18/18 tabel
7,Figur output lengkap,PASS,9/9 figur
8,Model tersimpan,PASS,3/3 model


NOTEBOOK 02 LOLOS.
Model siap dilanjutkan ke 03_Predictive_Modeling_GroupCV.ipynb
